# 02 — Why Time Series Breaks Your ML Pipeline (And How to Fix It)

**Companion notebook to Article 2 of the *Time Series Feature Engineering* series.**

📖 Read the article: [Medium](https://medium.com/@asidd24/why-time-series-breaks-your-ml-pipeline-and-how-to-fix-it-3177886faa54) · [LinkedIn Newsletter](https://www.linkedin.com/newsletters/time-series-engineered-7485187061080137728/)

---

This notebook demonstrates:

1. Why random train-test splits destroy time series models
2. Time-aware splitting and walk-forward cross-validation
3. How preprocessing pipelines leak future information — and how to fix them
4. Multi-step forecast strategies: recursive, direct, and multi-output
5. A complete production-grade pipeline putting it all together


## Setup

Install required packages and import shared utilities from the `src/` module.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error

from src.data import generate_daily_sales
from src.validation import make_walk_forward_splitter, describe_splits

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
np.random.seed(42)

## Generate Synthetic Data

We use the same synthetic daily sales series from Article 1 — 2 years of data
with trend, weekly + yearly seasonality, and Gaussian noise.

In [ ]:
sales = generate_daily_sales(n_days=730)
X = sales.to_frame(name='sales')
y = sales.copy()

print(f"Series length: {len(sales)} days")
print(f"Date range: {sales.index[0].date()} → {sales.index[-1].date()}")
sales.plot(title='Daily Sales — Synthetic Dataset', ylabel='Sales');

---

## Failure #1 — The Temporal Train-Test Split

The most common and damaging mistake: using `train_test_split()` on time series data.
Random shuffling lets the model see future data during training — a textbook case of **temporal leakage**.

### ❌ Wrong: Random (shuffled) split

In [ ]:
# ❌ This will give you a beautiful, meaningless score
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### ✅ Correct: Position-based (temporal) split

In [ ]:
# ✅ Split by position, preserving temporal order
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
# Verify: last train date should be before first test date
print(f"Last train date: {X_train.index[-1]}")
print(f"First test date: {X_test.index[0]}")

### Expanding-Window Cross-Validation with `TimeSeriesSplit`

Instead of K-Fold, use `TimeSeriesSplit` — each fold trains on data strictly before its test period.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    print(f"Fold {fold}: train {len(X_tr)} rows → test {len(X_te)} rows")

### Walk-Forward Validation with Rolling Window and Gap

In production, you want a **fixed** training window (not expanding) and a **gap** that
matches your forecast horizon — this prevents near-future data from leaking in.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
# Fixed 500-row training window, 100-row test window, gap of 12 for horizon
tscv = TimeSeriesSplit(
    n_splits=5,
    max_train_size=500,  # rolling window (not expanding)
    test_size=100,       # fixed test size
    gap=12              # separate train and test by forecast horizon
)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    print(f"Fold {fold}: train idx [{train_idx[0]}..{train_idx[-1]}], "
          f"test idx [{test_idx[0]}..{test_idx[-1]}], "
          f"gap = {test_idx[0] - train_idx[-1] - 1}")

### Using the `src.validation` utilities

The repo provides `make_walk_forward_splitter` and `describe_splits` to simplify this pattern.

In [ ]:
splitter = make_walk_forward_splitter(
    n_splits=5,
    max_train_size=500,
    test_size=100,
    forecast_horizon=12
)
describe_splits(splitter, X)

---

## Failure #2 — The Leaky Feature Pipeline

Even with correct splits, your preprocessing can still leak information.
Three common culprits:

### ❌ Leak 1: Scalers fit on the full dataset

The scaler learns the mean and standard deviation from *both* train and test data.

In [ ]:
# ❌ StandardScaler now knows the mean/std of the test set
scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X)  # <-- fits on train + test
X_train_scaled, X_test_scaled = X_all_scaled[:split_idx], X_all_scaled[split_idx:]

### ❌ Leak 2: Imputation with global statistics

Filling missing values with the mean of the entire dataset injects future information.

In [ ]:
# ❌ Same problem, different transform
X_imputed = X.fillna(X.mean())  # mean includes test period

### ❌ Leak 3: Rolling features computed on the full series

At time *t*, `rolling(7).mean()` includes `y(t)` itself. Fix: shift first.

In [ ]:
# ❌ At time t, rolling(7) window includes y(t) itself
df = X.copy()
df['rolling_mean_leaky'] = sales.rolling(7).mean()

# ✅ Fix: shift(1) so the window only uses past values
df['rolling_mean_safe'] = sales.shift(1).rolling(7).mean()

df[['sales', 'rolling_mean_leaky', 'rolling_mean_safe']].head(15)

### ✅ The Fix: `sklearn.pipeline.Pipeline`

Wrap preprocessing + model in a `Pipeline`. When used with `cross_val_score` and
`TimeSeriesSplit`, the scaler/imputer are fit *only* on each training fold.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor

pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
    ('model',  GradientBoostingRegressor(n_estimators=200, max_depth=4)),
])

# Now fit_transform is applied fold-safely inside cross_val_score
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
    pipe, X, y,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_mean_absolute_error'
)
print(f"CV MAE: {-scores.mean():.2f} ± {scores.std():.2f}")

### Pipeline with Feature-Engine Transformers

For time-series-specific feature creation (lags, rolling windows) inside the pipeline,
the [`feature-engine`](https://feature-engine.trainindata.com/) library integrates cleanly.

In [ ]:
from feature_engine.timeseries.forecasting import (
    LagFeatures,
    WindowFeatures
)

pipe = Pipeline([
    ('lags', LagFeatures(variables=['sales'], periods=[1, 7, 14])),
    ('window', WindowFeatures(
        variables=['sales'],
        window=7,
        functions=['mean', 'std'],
        freq='1D'
    )),
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
    ('model',  GradientBoostingRegressor(n_estimators=200, max_depth=4)),
])

---

## Failure #3 — The Multi-Step Forecast Trap

Most real-world applications need forecasts for multiple future steps (e.g., next 12 days).
There are three main strategies, each with different trade-offs.

### Strategy 1 — Recursive Forecasting

Train a single one-step model and feed its own predictions back as inputs.
Simple, but errors compound over time.

In [ ]:
def recursive_forecast(model, last_known_values, n_steps, lag_periods):
    """
    Forecast n_steps ahead by feeding predictions back as lag inputs.
    """
    forecasts = []
    history = list(last_known_values)  # rolling buffer of recent values
    for step in range(n_steps):
        # Build feature vector from current history
        features = [history[-lag] for lag in lag_periods]
        # Predict one step
        pred = model.predict([features])[0]
        forecasts.append(pred)
        # Append prediction to history for next step
        history.append(pred)
    return forecasts

### Strategy 2 — Direct Forecasting

Train one separate model per forecast horizon. Avoids error compounding
but requires maintaining multiple models.

In [ ]:
from sklearn.multioutput import MultiOutputRegressor

# Create separate targets for each horizon
def build_direct_targets(series, horizons):
    targets = pd.DataFrame({
        f'y_h{h}': series.shift(-h) for h in horizons
    })
    return targets.dropna()

y_direct = build_direct_targets(sales, horizons=range(1, 13))
X_aligned = X.loc[y_direct.index]

# Train one model per horizon
model = MultiOutputRegressor(
    GradientBoostingRegressor(n_estimators=100, max_depth=4)
)
model.fit(X_aligned, y_direct)

print(f"Training samples: {len(X_aligned)}")
print(f"Horizons: {y_direct.columns.tolist()}")

### Strategy 3 — Multi-Output Forecasting

Use a single model that natively outputs multiple values, capturing inter-step correlations.
Random Forests and neural networks support this natively.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# RandomForest natively supports multi-output regression
model = RandomForestRegressor(n_estimators=200, max_depth=6)
model.fit(X_aligned, y_direct)  # y_direct is (n_samples, 12)

# Quick sanity check
predictions = model.predict(X_aligned.tail(5))
print(f"Prediction shape: {predictions.shape}  (5 samples × 12 horizons)")

### Which strategy to use?

| Strategy | Pros | Cons |
|---|---|---|
| **Recursive** | Single model, simple | Error compounding |
| **Direct** | No compounding | Many models, less training data |
| **Multi-Output** | Captures inter-horizon correlations | Model must support multi-output |

---

## Putting It All Together — A Production-Grade Setup

This combines everything from the article:
- Multi-horizon target building
- Feature-engine transformers inside a leakage-safe pipeline
- Walk-forward cross-validation with a gap
- Per-horizon MAE evaluation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from feature_engine.timeseries.forecasting import LagFeatures, WindowFeatures
from lightgbm import LGBMRegressor

# 1. Build multi-horizon targets
horizons = list(range(1, 13))
y_multi = pd.DataFrame({
    f'h_{h}': sales.shift(-h)
    for h in horizons
}).dropna()
X_prod = sales.to_frame(name='sales').loc[y_multi.index]

# 2. Assemble leakage-safe pipeline
pipe = Pipeline([
    ('lags',    LagFeatures(variables=['sales'], periods=[1, 2, 3, 7, 14, 28])),
    ('window',  WindowFeatures(
        variables=['sales'],
        window=[7, 14],
        functions=['mean', 'std'],
        freq='1D'
    )),
    ('impute',  SimpleImputer(strategy='median')),
    ('scale',   StandardScaler()),
    ('model',   MultiOutputRegressor(
        LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, verbose=-1)
    )),
])

# 3. Walk-forward cross-validation with gap
tscv = TimeSeriesSplit(
    n_splits=5,
    max_train_size=730,  # 2 years of daily data
    test_size=90,        # 3 months
    gap=12               # match longest horizon
)

# 4. Evaluate per-horizon
from sklearn.metrics import mean_absolute_error
mae_by_horizon = np.zeros(len(horizons))

for train_idx, test_idx in tscv.split(X_prod):
    X_tr, X_te = X_prod.iloc[train_idx], X_prod.iloc[test_idx]
    y_tr, y_te = y_multi.iloc[train_idx], y_multi.iloc[test_idx]
    
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    
    for i, h in enumerate(horizons):
        mae_by_horizon[i] += mean_absolute_error(y_te.iloc[:, i], y_pred[:, i])

mae_by_horizon /= tscv.get_n_splits()

# 5. Report degradation with horizon
for h, mae in zip(horizons, mae_by_horizon):
    print(f"Horizon {h:2d}: MAE = {mae:.3f}")

### Visualize MAE Degradation by Horizon

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(horizons, mae_by_horizon, color='steelblue', edgecolor='white')
ax.set_xlabel('Forecast Horizon (days ahead)')
ax.set_ylabel('Mean Absolute Error')
ax.set_title('MAE Degradation by Forecast Horizon — Walk-Forward CV')
ax.set_xticks(horizons)
plt.tight_layout()
plt.show()

---

## Key Takeaways

| # | Rule |
|---|---|
| 1 | **Never** use `train_test_split()` on time series — split by position |
| 2 | Use `TimeSeriesSplit` with `gap` matching your forecast horizon |
| 3 | Wrap preprocessing + model in a `Pipeline` — scalers, imputers, and feature builders must be fit fold-safely |
| 4 | Always `shift(1)` before computing rolling features to avoid look-ahead leakage |
| 5 | Choose your multi-step strategy deliberately: recursive, direct, or multi-output |
| 6 | Report MAE **per horizon** — aggregate metrics hide degradation |


---

## What's Next

**Article 3** will cover **Decomposing Time Series: Separating Signal from Noise** — STL decomposition,
trend extraction, and using residuals as anomaly scores.

⭐ Star this repo to be notified when new articles drop.
